In [7]:
import numpy as np
import pandas as pd

np.random.seed(42)


In [10]:
n_weeks = 156  
dates = pd.date_range(start="2022-01-02", periods=n_weeks, freq="W-SUN")

df = pd.DataFrame({"week": dates})
df.head()


,week
0,2022-01-02
1,2022-01-09
2,2022-01-16
3,2022-01-23
4,2022-01-30


In [14]:
base = np.random.normal(loc=0, scale=1, size=n_weeks)

df["spend_search"]  = 40000 +  8000*base + np.random.normal(0, 6000, n_weeks)
df["spend_social"]  = 30000 +  7000*base + np.random.normal(0, 6000, n_weeks)
df["spend_display"] = 20000 +  5000*base + np.random.normal(0, 5000, n_weeks)
df["spend_ctv"]     = 50000 + 12000*base + np.random.normal(0, 9000, n_weeks)

for col in ["spend_search", "spend_social", "spend_display", "spend_ctv"]:
    df[col] = df[col].clip(lower=0);

df[["spend_search","spend_social","spend_display","spend_ctv"]].describe()


,spend_search,spend_social,spend_display,spend_ctv
count,156.000000,156.000000,156.000000,156.000000
mean,40066.477557,29919.179258,19798.259516,49083.813165
std,10685.325997,7883.237847,6518.822024,14778.485784
min,13666.047190,8374.996159,3196.119802,7774.455257
25%,33642.311462,25008.283538,15865.835288,40283.593996
50%,39554.135414,30098.502681,19705.779005,49168.380131
75%,47395.410676,34705.765274,24298.440269,59581.709392
max,70181.375695,49655.503844,39320.233447,90109.822599


In [28]:
df["week_of_year"] = df["week"].dt.isocalendar().week.astype(int)
df["seasonality_index"] = 1 + 0.20 * np.sin(2 * np.pi * df["week_of_year"] / 52)
df[["week", "week_of_year", "seasonality_index"]].head(10)

,week,week_of_year,seasonality_index
0,2022-01-02,52,1.000000
1,2022-01-09,1,1.024107
2,2022-01-16,2,1.047863
3,2022-01-23,3,1.070921
4,2022-01-30,4,1.092945
5,2022-02-06,5,1.113613
6,2022-02-13,6,1.132625
7,2022-02-20,7,1.149702
8,2022-02-27,8,1.164597
9,2022-03-06,9,1.177091


In [22]:
base_price = 50

df["price"] = (
    base_price
    + np.linspace(0, 3, n_weeks)                 
    + np.random.normal(0, 0.75, n_weeks)         
)

df["price"] = df["price"].clip(lower=35)

df[["week", "price"]].head(10)

,week,price
0,2022-01-02,50.391688
1,2022-01-09,49.945918
2,2022-01-16,51.743036
3,2022-01-23,50.724842
4,2022-01-30,50.507728
5,2022-02-06,49.139546
6,2022-02-13,49.149898
7,2022-02-20,50.355020
8,2022-02-27,50.263980
9,2022-03-06,49.713602


In [24]:
df["m_search"]  = np.log1p(df["spend_search"])
df["m_social"]  = np.log1p(df["spend_social"])
df["m_display"] = np.log1p(df["spend_display"])
df["m_ctv"]     = np.log1p(df["spend_ctv"])

In [25]:
df["m_ctv_lag1"] = df["m_ctv"].shift(1).fillna(df["m_ctv"].iloc[0])

In [30]:
base_revenue = 220000  

price_ref = df["price"].mean()
price_deviation = (df["price"] - price_ref)

w_search  = 18000
w_social  = 14000
w_display =  9000
w_ctv     = 22000

noise = np.random.normal(0, 12000, n_weeks)

df["revenue"] = (
    (base_revenue
     + w_search  * df["m_search"]
     + w_social  * df["m_social"]
     + w_display * df["m_display"]
     + w_ctv     * df["m_ctv_lag1"]
     - 2500      * price_deviation   
     + noise)
    * df["seasonality_index"]         
)

df["revenue"] = df["revenue"].clip(lower=0)

df[["week", "price", "seasonality_index", "revenue"]].head(10)


,week,price,seasonality_index,revenue
0,2022-01-02,50.391688,1.000000,8.935452e+05
1,2022-01-09,49.945918,1.024107,9.175149e+05
2,2022-01-16,51.743036,1.047863,9.503105e+05
3,2022-01-23,50.724842,1.070921,9.349959e+05
4,2022-01-30,50.507728,1.092945,9.529229e+05
5,2022-02-06,49.139546,1.113613,9.527438e+05
6,2022-02-13,49.149898,1.132625,1.029138e+06
7,2022-02-20,50.355020,1.149702,9.746074e+05
8,2022-02-27,50.263980,1.164597,1.033809e+06
9,2022-03-06,49.713602,1.177091,1.021092e+06


In [33]:
df[["spend_search","spend_social","spend_display","spend_ctv","price","seasonality_index","revenue"]].describe()

,spend_search,spend_social,spend_display,spend_ctv,price,seasonality_index,revenue
count,156.000000,156.000000,156.000000,156.000000,156.000000,156.000000,1.560000e+02
mean,40066.477557,29919.179258,19798.259516,49083.813165,51.491238,1.000000,8.784529e+05
std,10685.325997,7883.237847,6518.822024,14778.485784,1.221576,0.141877,1.263213e+05
min,13666.047190,8374.996159,3196.119802,7774.455257,48.228720,0.800000,6.693094e+05
25%,33642.311462,25008.283538,15865.835288,40283.593996,50.617579,0.863106,7.541838e+05
50%,39554.135414,30098.502681,19705.779005,49168.380131,51.417774,1.000000,8.927244e+05
75%,47395.410676,34705.765274,24298.440269,59581.709392,52.372812,1.136894,9.982963e+05
max,70181.375695,49655.503844,39320.233447,90109.822599,54.458095,1.200000,1.085757e+06


In [35]:
df.to_csv("../data/marketing_simulated_data.csv", index=False)